# Gemini Video Visual Question Answering

This notebook runs visual question answering on a video with the Gemini API. It supports local video files and YouTube URLs, and it uses the Gemini Files API for local videos so larger or reusable videos can be processed reliably.

Set `GEMINI_API_KEY` in your environment before running the notebook.

## 1. Install and Import Dependencies

Run the install cell once if `google-genai` is not already available in your notebook environment.

In [ ]:
%pip install -q google-genai python-dotenv ipywidgets

In [ ]:
import mimetypes
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, Video, display

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY or GOOGLE_API_KEY before running this notebook.")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-2.5-flash"

## 2. Choose a Video Source

Use either a local video path or a public YouTube URL. For local files, the notebook uploads the video and waits until Gemini finishes processing it.

In [ ]:
# Option A: local video file
VIDEO_PATH = Path("sample_video.mp4")  # change this to your video file

# Option B: public YouTube URL. Leave blank when using a local file.
YOUTUBE_URL = ""  # example: "https://www.youtube.com/watch?v=9hE5-98ZeCg"

if YOUTUBE_URL:
    display(Markdown(f"Using YouTube URL: {YOUTUBE_URL}"))
elif VIDEO_PATH.exists():
    display(Video(str(VIDEO_PATH), embed=False, width=720))
else:
    display(Markdown(f"Update `VIDEO_PATH`; file not found: `{VIDEO_PATH}`"))

## 3. Prepare the Video for Gemini

In [ ]:
SUPPORTED_VIDEO_MIME_TYPES = {
    "video/mp4",
    "video/mpeg",
    "video/mov",
    "video/avi",
    "video/x-flv",
    "video/mpg",
    "video/webm",
    "video/wmv",
    "video/3gpp",
}


def guess_video_mime_type(path: Path) -> str:
    mime_type, _ = mimetypes.guess_type(path)
    if mime_type in SUPPORTED_VIDEO_MIME_TYPES:
        return mime_type
    if path.suffix.lower() == ".mov":
        return "video/mov"
    raise ValueError(
        f"Unsupported or unknown video MIME type for {path}. "
        f"Supported types: {', '.join(sorted(SUPPORTED_VIDEO_MIME_TYPES))}"
    )


def wait_for_file_active(uploaded_file, poll_seconds: int = 5, timeout_seconds: int = 600):
    start = time.time()
    file_obj = uploaded_file

    while not file_obj.state or file_obj.state.name != "ACTIVE":
        if file_obj.state and file_obj.state.name == "FAILED":
            raise RuntimeError(f"Gemini file processing failed: {file_obj.name}")
        if time.time() - start > timeout_seconds:
            raise TimeoutError(f"Timed out waiting for Gemini to process: {file_obj.name}")

        print(f"Processing video... state={file_obj.state}")
        time.sleep(poll_seconds)
        file_obj = client.files.get(name=file_obj.name)

    print(f"Video ready: {file_obj.name}")
    return file_obj


def prepare_video_input(video_path: Path | None = None, youtube_url: str = ""):
    if youtube_url:
        return types.Part(file_data=types.FileData(file_uri=youtube_url))

    if not video_path or not video_path.exists():
        raise FileNotFoundError("Provide an existing local video path or a public YouTube URL.")

    mime_type = guess_video_mime_type(video_path)
    upload_config = types.UploadFileConfig(mime_type=mime_type)
    uploaded = client.files.upload(file=video_path, config=upload_config)
    return wait_for_file_active(uploaded)


video_input = prepare_video_input(None if YOUTUBE_URL else VIDEO_PATH, YOUTUBE_URL)

## 4. Ask Visual Questions About the Video

The prompt asks Gemini to answer only from visible or audible evidence and to include timestamps where useful.

In [ ]:
def ask_video_question(question: str, *, fps: float | None = None) -> str:
    prompt = f"""
You are a careful video visual question answering assistant.
Answer the user's question using only evidence from the video.
Mention relevant timestamps when they help support the answer.
If the video does not contain enough evidence, say that clearly.

Question: {question}
""".strip()

    if fps is not None and not YOUTUBE_URL:
        video_part = types.Part(
            file_data=types.FileData(file_uri=video_input.uri, mime_type=video_input.mime_type),
            video_metadata=types.VideoMetadata(fps=fps),
        )
    else:
        video_part = video_input

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=[video_part, prompt],
    )
    return response.text


question = "What are the main actions or events in this video?"
answer = ask_video_question(question)
display(Markdown(answer))

## 5. Ask Multiple Questions

In [ ]:
questions = [
    "Describe the scene and important objects visible in the video.",
    "What changes over time from the beginning to the end?",
    "Are there any people, vehicles, animals, text, or notable sounds?",
    "What happens around 00:05, if that timestamp exists in the video?",
]

for idx, q in enumerate(questions, start=1):
    print(f"Question {idx}: {q}")
    display(Markdown(ask_video_question(q)))

## 6. Optional: Timestamped Structured Output

Use this when you want a concise table-like answer for downstream analysis.

In [ ]:
structured_prompt = """
Create a concise markdown table with these columns:
- timestamp
- visual evidence
- audio evidence, if any
- answer-relevant observation

Focus on details that help answer: What is happening in the video?
""".strip()

display(Markdown(ask_video_question(structured_prompt)))

## 7. Clean Up Uploaded Files

Run this if you do not need to reuse the uploaded local video in later prompts.

In [ ]:
if not YOUTUBE_URL and hasattr(video_input, "name"):
    client.files.delete(name=video_input.name)
    print(f"Deleted uploaded file: {video_input.name}")

## Notes

- Use the Files API for local videos larger than 20 MB, videos longer than roughly a minute, or videos you want to query multiple times.
- For short local videos under 20 MB, inline upload can also work, but the Files API path is more reusable.
- Gemini samples video frames by default and can use timestamps like `MM:SS` in prompts.
- Increase `fps` in `ask_video_question(question, fps=...)` for fast-moving videos, or lower it for long, mostly static videos.